# Phase 8 — Fine-tune from SimCLR-Pretrained Backbone

Same heteroscedastic architecture and training recipe as notebook 05, but `self.backbone` is initialized from `simclr_pretrain_v1/backbone_only.pt` (notebook 07) instead of plain ImageNet weights.

**Two things this notebook tests, both needed for the paper's novelty claim:**
1. **Full-label comparison**: does self-supervised pretraining improve val_mae/val_auc over the notebook 05 ImageNet-only baseline, at the same (full) amount of labeled data?
2. **Label-efficiency comparison**: does the pretrained backbone need *less* labeled data to reach similar accuracy? We fine-tune on 25%, 50%, and 100% of the TRAIN split and compare both init strategies at each fraction — this is the actual 'semi-supervised helps when labels are scarce' claim from the roadmap's novelty audit, not just a single-point comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, mean_absolute_error

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
FRAME_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/full', 'frames')
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')
PRETRAINED_BACKBONE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'simclr_pretrain_v1', 'backbone_only.pt')

assert os.path.exists(PRETRAINED_BACKBONE_PATH), 'Run 07_selfsupervised_pretraining.ipynb first.'

df = pd.read_csv(FULL_CSV)
quality_df = pd.read_csv(quality_csv_path)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
HFrEF_THRESHOLD = 40

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(FRAME_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask].reset_index(drop=True)
train_df_full = clean_df[clean_df['Split'].str.upper() == 'TRAIN'].reset_index(drop=True)
val_df = clean_df[clean_df['Split'].str.upper() == 'VAL'].reset_index(drop=True)
print(f'Full train: {len(train_df_full)}  Val: {len(val_df)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU before running multiple fine-tune experiments.')

In [ ]:
# Reproducibility: fix random seeds (used for all runs from this version onward)
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Note: GPU kernels (cuDNN) can still be non-deterministic, so re-runs may differ slightly in the last digits.

In [ ]:
class EchoFrameDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        aug = [T.RandomHorizontalFlip()] if train else []
        self.transform = T.Compose([T.ToPILImage(), T.Resize((112, 112)), *aug, T.ToTensor(), T.Normalize(mean, std)])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        frame = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        img = self.transform(frame)
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        return img, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

val_ds = EchoFrameDataset(val_df, FRAME_CACHE_DIR, filename_col, ef_col, train=False)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

class EchoHeteroscedastic(nn.Module):
    def __init__(self, dropout_p=0.2, pretrained_backbone_path=None):
        super().__init__()
        if pretrained_backbone_path:
            backbone = models.resnet18(weights=None)
            backbone.fc = nn.Identity()
            backbone.load_state_dict(torch.load(pretrained_backbone_path, map_location='cpu'))
        else:
            backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            backbone.fc = nn.Identity()
        n_feat = 512  # resnet18 final feature dim
        self.backbone = backbone
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_mean_head = nn.Linear(n_feat, 1)
        self.ef_logvar_head = nn.Linear(n_feat, 1)
        self.cls_head = nn.Linear(n_feat, 1)

    def forward(self, x):
        feat = self.dropout(self.backbone(x))
        return self.ef_mean_head(feat).squeeze(-1), self.ef_logvar_head(feat).squeeze(-1), self.cls_head(feat).squeeze(-1)

def gaussian_nll_loss(ef_true, ef_mean, ef_logvar):
    ef_logvar = ef_logvar.clamp(min=-6, max=6)
    return (0.5 * torch.exp(-ef_logvar) * (ef_true - ef_mean) ** 2 + 0.5 * ef_logvar).mean()

def evaluate(model, loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for imgs, ef_true, cls_true in loader:
            imgs = imgs.to(device)
            ef_mean, ef_logvar, cls_logit = model(imgs)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_mean.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    f1 = f1_score(cls_true_all, [1 if p > 0.5 else 0 for p in cls_prob_all], zero_division=0)
    return {'mae': mae, 'auc': auc, 'f1': f1}

## Reusable fine-tune function (for both init strategies, at any label fraction)

In [ ]:
def finetune_run(exp_name, label_fraction, use_pretrained, n_epochs=4, checkpoint_every_minutes=15):
    if label_fraction < 1.0:
        train_df = train_df_full.sample(frac=label_fraction, random_state=42).reset_index(drop=True)
    else:
        train_df = train_df_full
    train_ds = EchoFrameDataset(train_df, FRAME_CACHE_DIR, filename_col, ef_col, train=True)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)

    backbone_path = PRETRAINED_BACKBONE_PATH if use_pretrained else None
    model = EchoHeteroscedastic(pretrained_backbone_path=backbone_path).to(device)
    with torch.no_grad():
        mean_ef = float(train_df[ef_col].mean())
        model.ef_mean_head.bias.fill_(mean_ef)
        model.ef_mean_head.weight.zero_()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    pos_rate = (train_df[ef_col] < HFrEF_THRESHOLD).mean()
    pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
    cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    state_path = os.path.join(PROJECT_ROOT, 'checkpoints', exp_name, 'latest.pt')
    os.makedirs(os.path.dirname(state_path), exist_ok=True)
    start_epoch, resume_step = 0, 0
    if os.path.exists(state_path):
        state = torch.load(state_path, map_location=device)
        model.load_state_dict(state['model_state']); optimizer.load_state_dict(state['optimizer_state'])
        start_epoch, resume_step = state['epoch'], state['step_in_epoch']
        print(f'[{exp_name}] resumed from epoch={start_epoch} step={resume_step}')

    last_ckpt_time = time.time()
    for epoch in range(start_epoch, n_epochs):
        model.train()
        for step, (imgs, ef_true, cls_true) in enumerate(train_loader):
            if epoch == start_epoch and step < resume_step:
                continue
            imgs, ef_true, cls_true = imgs.to(device), ef_true.to(device), cls_true.to(device)
            ef_mean, ef_logvar, cls_logit = model(imgs)
            loss = gaussian_nll_loss(ef_true, ef_mean, ef_logvar) + cls_criterion(cls_logit, cls_true)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            if (time.time() - last_ckpt_time) > checkpoint_every_minutes * 60:
                torch.save({'epoch': epoch, 'step_in_epoch': step + 1, 'model_state': model.state_dict(),
                            'optimizer_state': optimizer.state_dict()}, state_path)
                last_ckpt_time = time.time()
        resume_step = 0
        torch.save({'epoch': epoch + 1, 'step_in_epoch': 0, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict()}, state_path)
        val_metrics = evaluate(model, val_loader)
        print(f"[{exp_name}] epoch {epoch}: val_mae={val_metrics['mae']:.2f} val_auc={val_metrics['auc']:.3f}")

    final_metrics = evaluate(model, val_loader)
    return final_metrics

## Run the comparison: ImageNet-only vs. SimCLR-pretrained, at 3 label fractions

This is 6 fine-tune runs total. Each is short (`n_epochs=4`) since we're comparing *relative* performance, not chasing a final number yet. If a session disconnects mid-way, just re-run this cell -- each individual `finetune_run` call resumes from its own checkpoint, and completed runs will simply repeat fast (already at n_epochs, so they just re-evaluate).

In [ ]:
results = {}
for frac in [0.25, 0.5, 1.0]:
    for use_pretrained in [False, True]:
        tag = 'simclr' if use_pretrained else 'imagenet'
        exp_name = f'finetune_{tag}_frac{int(frac*100)}'
        print(f'--- Running {exp_name} ---')
        metrics = finetune_run(exp_name, frac, use_pretrained)
        results[exp_name] = metrics
        print(f'{exp_name}: {metrics}')

results_df = pd.DataFrame(results).T
results_df.index.name = 'experiment'
print(results_df)
results_df.to_csv(os.path.join(PROJECT_ROOT, 'logs', 'label_efficiency_comparison.csv'))

## Plot: label efficiency curve, ImageNet-only vs. SimCLR-pretrained

In [ ]:
import matplotlib.pyplot as plt

fracs = [25, 50, 100]
imagenet_mae = [results[f'finetune_imagenet_frac{f}']['mae'] for f in fracs]
simclr_mae = [results[f'finetune_simclr_frac{f}']['mae'] for f in fracs]
imagenet_auc = [results[f'finetune_imagenet_frac{f}']['auc'] for f in fracs]
simclr_auc = [results[f'finetune_simclr_frac{f}']['auc'] for f in fracs]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fracs, imagenet_mae, marker='o', label='ImageNet-only init')
axes[0].plot(fracs, simclr_mae, marker='o', label='SimCLR-pretrained init')
axes[0].set_xlabel('% of TRAIN labels used'); axes[0].set_ylabel('EF MAE (val)')
axes[0].set_title('Label efficiency: EF regression'); axes[0].legend()

axes[1].plot(fracs, imagenet_auc, marker='o', label='ImageNet-only init')
axes[1].plot(fracs, simclr_auc, marker='o', label='SimCLR-pretrained init')
axes[1].set_xlabel('% of TRAIN labels used'); axes[1].set_ylabel('HFrEF AUC (val)')
axes[1].set_title('Label efficiency: HFrEF classification'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'figures', 'label_efficiency_curve.png'), dpi=150)
plt.show()

## What to check

- **If SimCLR-pretrained beats ImageNet-only, especially at 25%/50% labels**: this is the label-efficiency result the novelty audit was hoping for -- a genuine, reportable finding supporting the 'semi-supervised helps in the label-scarce regime' claim.
- **If they're roughly tied, or ImageNet-only wins**: also a legitimate, honest finding -- report it as such rather than reframing. It would suggest ImageNet features already transfer well to echo, and the self-supervised pretraining's main value may lie elsewhere (e.g., the general representation quality) rather than label efficiency specifically. Either outcome is publishable; the point of this experiment was to test the claim, not assume it.
- Only 4 epochs per fine-tune run here (for speed, comparing 6 runs). Once a clear winner emerges, that single best configuration should get a longer, final fine-tune before the test-set evaluation.